<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB05_Matplotlib_Fundamentals_Figures_Axes_and_Real_Naval_Plots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB05 · Class 5 — Matplotlib Fundamentals: Figures, Axes, and Real Naval Plots**

## Block 2: AI — Machine Learning (continued)

Every class since `NB02` has plotted something — a scatter plot, a confusion matrix, a loss curve, a geographic map — but always as a side effect of whatever concept was being taught, never as its own subject. This class is the first systematic look at Matplotlib itself: the Figure/Axes object model every plot in this course is quietly built on, real styling and layout control, and colormap plots (the direct foundation for `NB18`'s and `NB20`'s geographic maps). Every example here uses the real `ship_fuel_efficiency.csv` (`NB07`) or `sonar.all-data` (`NB08`) — no synthetic numbers.

### Learning objectives

By the end of this class, students will be able to:
- Explain the difference between a Matplotlib `Figure` and an `Axes`, and why almost every real plot should be built through `fig, ax = plt.subplots()` rather than the older `plt.plot()` shortcut style.
- Style a plot deliberately (color, linestyle, marker, labels) instead of accepting whatever Matplotlib picks by default.
- Add legends and annotations that make a plot self-explanatory without a caption.
- Lay out multiple related plots in one figure with `plt.subplots(rows, cols)`.
- Build a colormap plot (`imshow`) from a real 2-D array.
- Use a twin axis to show two real quantities with different units on the same plot.
- Save a figure to a real file at publication quality.

### Agenda (2-hour class)

| # | Section | Minutes |
|---|---|---|
| 1 | Recap and why a systematic Matplotlib class | 5 |
| 2 | The Figure/Axes object model | 15 |
| 3 | Styling: color, linestyle, marker, labels | 15 |
| 4 | Legends and annotations | 15 |
| 5 | Subplot layouts | 20 |
| 6 | Colormap plots on real data | 20 |
| 7 | Twin axes | 15 |
| 8 | Saving figures | 5 |
| 9 | Summary, homework, next class | 10 |

As always: approximate guidance, not a script.


---

## 1. Why a systematic Matplotlib class


Every plot in this course so far worked, but was written slightly differently each time, borrowing whatever pattern the moment needed. This class collects that scattered knowledge into one coherent model, so every later notebook's plotting code (including `NB18`'s and `NB20`'s geographic maps) is something you already understand, not something to copy without full understanding.

> **Further reading**: [Matplotlib official documentation](https://matplotlib.org/stable/) | [Matplotlib (Wikipedia)](https://en.wikipedia.org/wiki/Matplotlib)


---

## 2. The Figure/Axes object model


A Matplotlib **`Figure`** is the whole window or saved image; an **`Axes`** is one actual plot living inside it (despite the confusingly similar name, an `Axes` is not the same thing as the x/y axis lines). A figure can hold several axes (Section 5). The recommended pattern — used throughout this course from here on — is to create both explicitly with `plt.subplots()` and call methods directly on the `ax` object, rather than the older `plt.plot()`/`plt.xlabel()` global-state style, which becomes ambiguous the moment a figure has more than one subplot.


In [ ]:
import matplotlib.pyplot as plt
import urllib.request
import csv

fuel_url = "https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/ship_fuel_efficiency.csv"
rows = list(csv.DictReader(urllib.request.urlopen(fuel_url).read().decode("utf-8").splitlines()))

distance = [float(r["distance"]) for r in rows]
fuel = [float(r["fuel_consumption"]) for r in rows]

fig, ax = plt.subplots(figsize=(7, 5))   # fig = the whole figure, ax = the one plot inside it
ax.scatter(distance, fuel, s=8, alpha=0.5)
ax.set_xlabel("Distance (nautical miles)")
ax.set_ylabel("Fuel consumption")
ax.set_title(f"Real data: {len(rows)} voyages from ship_fuel_efficiency.csv")
plt.show()


> **Further reading**: [Matplotlib Figure/Axes quickstart](https://matplotlib.org/stable/users/explain/quick_start.html) | [`matplotlib.pyplot.subplots` documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html)


---

## 3. Styling: color, linestyle, marker, labels


One real ship's 12 monthly `fuel_consumption` readings (already used this way in `NB14`'s LSTM forecasting) make a clean real line plot to style deliberately: a specific color, a dashed line, circular markers at each real data point, and readable axis labels.


In [ ]:
ship_id = "NG001"
ship_rows = [r for r in rows if r["ship_id"] == ship_id]   # already in Jan-Dec order in the real file
months = [r["month"][:3] for r in ship_rows]
ship_fuel = [float(r["fuel_consumption"]) for r in ship_rows]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(months, ship_fuel, color="darkorange", linestyle="--", marker="o", markersize=6, linewidth=1.5)
ax.set_xlabel("Month")
ax.set_ylabel("Fuel consumption")
ax.set_title(f"Real monthly fuel consumption -- ship {ship_id}")
ax.grid(alpha=0.3)
plt.show()


> **Further reading**: [Matplotlib named colors](https://matplotlib.org/stable/gallery/color/named_colors.html) | [Matplotlib line/marker styles](https://matplotlib.org/stable/api/markers_api.html)


---

## 4. Legends and annotations


A plot with more than one series needs a **legend** to stay readable, and an **annotation** can point directly at a specific real value worth calling out — here, comparing three real ships' monthly fuel consumption and marking the single highest real reading among them.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

max_point = (None, None, None, -1)  # ship_id, month_label, value, index -- track the real overall max
for sid in ["NG001", "NG002", "NG003"]:
    s_rows = [r for r in rows if r["ship_id"] == sid]
    s_months = [r["month"][:3] for r in s_rows]
    s_fuel = [float(r["fuel_consumption"]) for r in s_rows]
    ax.plot(s_months, s_fuel, marker="o", markersize=4, label=sid)
    local_max_idx = s_fuel.index(max(s_fuel))
    if max_point[2] is None or s_fuel[local_max_idx] > max_point[2]:
        max_point = (sid, s_months[local_max_idx], s_fuel[local_max_idx], local_max_idx)

ax.annotate(
    f"Highest reading: {max_point[0]}, {max_point[1]} ({max_point[2]:.0f})",
    xy=(max_point[3], max_point[2]),
    xytext=(max_point[3] + 1, max_point[2] + 300),
    arrowprops=dict(arrowstyle="->", color="black"),
)
ax.set_xlabel("Month")
ax.set_ylabel("Fuel consumption")
ax.set_title("Real monthly fuel consumption -- 3 ships")
ax.legend(title="Ship ID")
plt.show()


> **Further reading**: [Matplotlib legend guide](https://matplotlib.org/stable/users/explain/axes/legend_guide.html) | [Matplotlib annotation guide](https://matplotlib.org/stable/users/explain/text/annotations.html)


---

## 5. Subplot layouts


`plt.subplots(rows, cols)` returns a whole grid of `Axes` at once. The cell below builds a real 2x2 dashboard from the fuel dataset: a histogram, a boxplot by ship type, a scatter plot, and a bar chart of per-type averages — four different real views of the same dataset in one figure.


In [ ]:
import numpy as np

ship_types = sorted(set(r["ship_type"] for r in rows))
fuel_by_type = {t: [float(r["fuel_consumption"]) for r in rows if r["ship_type"] == t] for t in ship_types}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].hist(fuel, bins=25, color="steelblue", edgecolor="white")
axes[0, 0].set_title("Distribution of fuel consumption")

axes[0, 1].boxplot([fuel_by_type[t] for t in ship_types], tick_labels=ship_types)
axes[0, 1].set_title("Fuel consumption by ship type")
axes[0, 1].tick_params(axis="x", rotation=30)

axes[1, 0].scatter(distance, fuel, s=6, alpha=0.4, color="seagreen")
axes[1, 0].set_title("Distance vs. fuel consumption")
axes[1, 0].set_xlabel("Distance")

axes[1, 1].bar(ship_types, [np.mean(fuel_by_type[t]) for t in ship_types], color="coral")
axes[1, 1].set_title("Average fuel consumption per type")
axes[1, 1].tick_params(axis="x", rotation=30)

fig.suptitle("Four real views of the same dataset, one figure", fontsize=13)
plt.tight_layout()
plt.show()


For layouts where the panels aren't a uniform grid (e.g. one wide plot on top, two narrower ones below), `plt.subplots` alone can't express that — `matplotlib.gridspec.GridSpec` can.


In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(9, 6))
gs = gridspec.GridSpec(2, 2, figure=fig)

ax_top = fig.add_subplot(gs[0, :])         # spans both columns, top row
ax_bottom_left = fig.add_subplot(gs[1, 0])
ax_bottom_right = fig.add_subplot(gs[1, 1])

ax_top.plot(months, ship_fuel, marker="o")
ax_top.set_title(f"Wide top panel: ship {ship_id} monthly fuel")

ax_bottom_left.hist(fuel_by_type["Tanker Ship"], bins=15, color="slateblue")
ax_bottom_left.set_title("Tanker Ship distribution")

ax_bottom_right.hist(fuel_by_type["Fishing Trawler"], bins=15, color="darkgoldenrod")
ax_bottom_right.set_title("Fishing Trawler distribution")

plt.tight_layout()
plt.show()


> **Further reading**: [Matplotlib subplots guide](https://matplotlib.org/stable/users/explain/axes/arranging_axes.html) | [`matplotlib.gridspec` documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.gridspec.GridSpec.html)


---

## 6. Colormap plots on real data


`imshow` renders a 2-D array directly as a colored grid — the exact plotting call `NB18`'s ERA5 grids and `NB20`'s bathymetry grid both build on, before adding real geographic coastlines with `cartopy` on top. Here, it visualizes something already real and already computed in this course: the full 60x60 correlation matrix between every pair of Sonar frequency bands (`NB08`).


In [ ]:
sonar_url = "https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data"
sonar_lines = urllib.request.urlopen(sonar_url).read().decode("utf-8").strip().split("\n")
sonar_features = np.array([[float(x) for x in line.split(",")[:-1]] for line in sonar_lines])

corr_matrix = np.corrcoef(sonar_features.T)   # real 60x60 correlation matrix

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xlabel("Frequency band index")
ax.set_ylabel("Frequency band index")
ax.set_title("Real correlation matrix -- 60 Sonar frequency bands")
plt.colorbar(im, ax=ax, label="Correlation")
plt.show()


The bright diagonal (always exactly 1.0 -- every band perfectly correlates with itself) and the broad red band near it (neighboring frequency bands correlate strongly with each other) are both real, physically expected patterns: adjacent sonar frequencies respond to the same underlying echo, so they rise and fall together.

> **Further reading**: [`matplotlib.pyplot.imshow` documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) | [Matplotlib colormap reference](https://matplotlib.org/stable/users/explain/colors/colormaps.html)


---

## 7. Twin axes


Two real quantities with genuinely different scales and units — `distance` (nautical miles) and `fuel_consumption` (its own units) for one real ship's 12 months — are hard to compare on one shared y-axis. `ax.twinx()` creates a second y-axis sharing the same x-axis, so both series can be read on their own natural scale.


In [ ]:
ship_distance = [float(r["distance"]) for r in ship_rows]

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax2 = ax1.twinx()

line1, = ax1.plot(months, ship_distance, color="steelblue", marker="o", label="Distance")
line2, = ax2.plot(months, ship_fuel, color="darkorange", marker="s", label="Fuel consumption")

ax1.set_xlabel("Month")
ax1.set_ylabel("Distance (nm)", color="steelblue")
ax2.set_ylabel("Fuel consumption", color="darkorange")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax2.tick_params(axis="y", labelcolor="darkorange")
ax1.set_title(f"Real distance and fuel consumption together -- ship {ship_id}")
ax1.legend(handles=[line1, line2], loc="upper left")
plt.show()


> **Further reading**: [Matplotlib twin axes guide](https://matplotlib.org/stable/gallery/subplots_axes_and_figures/two_scales.html)


---

## 8. Saving figures


`fig.savefig(...)` writes the real figure to a file at a chosen resolution (`dpi`) and format (PNG, PDF, SVG...). This matters the moment a plot needs to go into a report rather than just a notebook cell — the default on-screen resolution is usually too low for print.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(months, ship_fuel, marker="o", color="darkorange")
ax.set_title(f"Ship {ship_id} -- saved at publication quality")
fig.savefig("ship_fuel_plot.png", dpi=200, bbox_inches="tight")
print("Saved ship_fuel_plot.png")


> **Further reading**: [`Figure.savefig` documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.figure.Figure.savefig.html)


---

## Class summary

- Build every real plot through `fig, ax = plt.subplots()`, not the older global-state `plt.plot()` style — it stays unambiguous the moment a figure has more than one subplot.
- Deliberate styling (color, linestyle, marker) and labels make a plot self-explanatory instead of relying on a caption to explain it.
- Legends and annotations are how a multi-series or single-highlight plot stays readable.
- `plt.subplots(rows, cols)` covers uniform grids; `GridSpec` covers layouts that aren't uniform.
- `imshow` turns any real 2-D array into a colormap plot — the direct foundation for `NB18`'s and `NB20`'s geographic maps.
- Twin axes let two real quantities with different units share one x-axis honestly, each on its own scale.

## For the next class (NB06)

SciPy for naval engineering: numerical differentiation, integration, interpolation, and optimization, applied to real ship performance data — including a real interpolation of the sparse `NB10` Yacht Hydrodynamics towing-tank measurements.

## Homework / Practice Ideas

1. Rebuild Section 2's scatter plot colored by `ship_type` (hint: one `ax.scatter()` call per type, each with its own color and a legend).
2. Add a second `twinx()` axis idea of your own: plot `engine_efficiency` against `CO2_emissions` for one real ship's 12 months.
3. Extend Section 5's 2x2 dashboard with a fifth real view using `GridSpec` instead of a uniform grid.
4. Build a real correlation-matrix `imshow` plot (Section 6's technique) for the numeric columns of `ship_fuel_efficiency.csv` instead of the Sonar dataset -- what real correlations show up (e.g. distance vs. fuel consumption)?
5. Save one of this class's real figures as both a `.png` (`dpi=200`) and a `.pdf`, and compare the real file sizes -- why does the difference make sense given what each format actually stores?

> ***As always: a plot's real job is to make a real pattern easier to see than the raw numbers alone -- deliberate styling and layout are part of that job, not decoration.***
